##### This is for generating WordClouds based on the NGRAMS API and using Z-scores

In [87]:
#pip install stopwordsiso

In [88]:
#importing libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import time
from stopwordsiso import stopwords
import re

In [89]:
#loading the data

df = pd.read_csv('evaluation_results3.csv')
df.head(5)

,ID,Data Provider,Project Name,Consumer Team,Consumer Name,Consumer Description,Variation Type,Variation Value,Purpose,Realistic?,Decision,AI Decision 1,AI Decision 2,AI Decision 3,AI Warning 1,AI Warning 2,AI Warning 3,Vote Count,Final AI Decision
0,8392a6d9159cc8a43f7794b73694e375edddcaecc8afb2...,Benefit,Claims Risk Pattern Analysis,underwriting,Risk Insight Tool,A platform designed to offer in-depth risk ins...,NaN,Original Request,"To improve our risk assessment process, we aim...","Yes, this is a realistic access request for a ...",Accept,Accept,Accept,Accept,NaN,NaN,NaN,3-0,Accept
1,8392a6d9159cc8a43f7794b73694e375edddcaecc8afb2...,Benefit,Claims Risk Pattern Analysis,underwriting,Risk Insight Tool,A platform designed to offer in-depth risk ins...,combined,"intern + very hasty (typos, shorthand, missing...",hey need access to old claims data asap gotta ...,NaN,NaN,Reject,Reject,Reject,"{'title': 'Default policies', 'policyKey': 'gl...","{'policyKey': 'Default policies', 'title': 'Da...","{'policyKey': 'default', 'title': 'Default pol...",0-3,Reject
2,8392a6d9159cc8a43f7794b73694e375edddcaecc8afb2...,Benefit,Claims Risk Pattern Analysis,underwriting,Risk Insight Tool,A platform designed to offer in-depth risk ins...,combined,"intern + neutral (standard professional, no pa...",I'm looking to access historical claims data t...,NaN,NaN,Accept,Accept,Accept,NaN,NaN,NaN,3-0,Accept
3,8392a6d9159cc8a43f7794b73694e375edddcaecc8afb2...,Benefit,Claims Risk Pattern Analysis,underwriting,Risk Insight Tool,A platform designed to offer in-depth risk ins...,combined,intern + very formal (precise legal-style lang...,I hereby formally request access to the histor...,NaN,NaN,Accept,Accept,Accept,NaN,NaN,NaN,3-0,Accept
4,8392a6d9159cc8a43f7794b73694e375edddcaecc8afb2...,Benefit,Claims Risk Pattern Analysis,underwriting,Risk Insight Tool,A platform designed to offer in-depth risk ins...,combined,"junior analyst + very hasty (typos, shorthand,...",hey can I get access to the old claims data? n...,NaN,NaN,Reject,Reject,Reject,"{'policyKey': 'Default policies', 'title': 'Da...","{'policyKey': 'Default policies', 'title': 'Da...","{'policyKey': 'Default policies', 'title': 'Da...",0-3,Reject


In [90]:
#extracting columns for the seniority and the hastiness dimension.

def extracting_seniority(text):
    text = str(text).lower()

    if "intern" in text:
        return "Intern"
    elif "junior analyst" in text:
        return "Junior Analyst"
    elif "senior manager" in text:
        return "Senior Manager"
    elif "executive/ceo" in text:
        return "Executive/CEO"
    #elif "original request" in text:
    #    return "Original"
    return None


def extracting_hastiness(text):
    text = str(text).lower()

    if "very hasty" in text:
        return "Very Hasty"
    elif "neutral" in text:
        return "Neutral"
    elif "very formal" in text:
        return "Very Formal"
    #elif "original request" in text:
        #return "Original"
    return None

df['Seniority'] = df['Variation Value'].apply(extracting_seniority)
df['Hastiness'] = df['Variation Value'].apply(extracting_hastiness)



seniority_category=["Intern", "Junior Analyst", "Senior Manager", "Executive/CEO", "Original"]
hastiness_category=["Very Hasty", "Neutral", "Very Formal", "Original"]

df["Seniority"] = pd.Categorical(df["Seniority"],categories = seniority_category, ordered=True)
df["Hastiness"] = pd.Categorical(df["Hastiness"],categories = hastiness_category, ordered=True)



In [91]:
#cleaning the text and tokenising it (removing punctuation etc.)

def tokenise(text):
    if pd.isna(text):
        return []
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", "", text) #removing any punctuation from the prompt
    text = re.sub(r"http\S+", "", text) #removing any URLs from the prompt if there's any
    text = re.sub(r"\s+", " ", text) #removing any extra whitespace

    words = text.split() #splitting the text into words

    return words

#applying the tokenis into the purpose column
df["Tokens"] = (df["Purpose"].apply(tokenise))

#checking if it works
df[['Purpose', 'Tokens']].head(5)

,Purpose,Tokens
0,"To improve our risk assessment process, we aim...","[to, improve, our, risk, assessment, process, ..."
1,hey need access to old claims data asap gotta ...,"[hey, need, access, to, old, claims, data, asa..."
2,I'm looking to access historical claims data t...,"[im, looking, to, access, historical, claims, ..."
3,I hereby formally request access to the histor...,"[i, hereby, formally, request, access, to, the..."
4,hey can I get access to the old claims data? n...,"[hey, can, i, get, access, to, the, old, claim..."


In [92]:
print(df["Tokens"].iloc[1])

['hey', 'need', 'access', 'to', 'old', 'claims', 'data', 'asap', 'gotta', 'check', 'out', 'risk', 'patterns', 'tied', 'to', 'diff', 'groups', 'and', 'actions', 'wanna', 'use', 'beneficiaries', 'info', 'to', 'make', 'risk', 'profiles', 'for', 'various', 'customer', 'types', 'thisll', 'help', 'us', 'tweak', 'underwriting', 'and', 'predict', 'premiums', 'better', 'for', 'upcoming', 'policies', 'goal', 'is', 'to', 'make', 'sure', 'our', 'policies', 'match', 'real', 'risk', 'so', 'it', 'helps', 'the', 'company', 'with', 'better', 'financial', 'guesses', 'and', 'gives', 'customers', 'fair', 'insurance', 'prices', 'thx']


In [93]:
#calculating the word frequencies, using the Counter library to count the words

from collections import Counter

def get_word_frequencies(df):
    all_words = []

    for tokens in df["Tokens"]:
        all_words.extend(tokens)

    word_counts = Counter(all_words)

    frequency_df = pd.DataFrame(word_counts.items(),columns=["Word", "Count"])

    total_words = frequency_df["Count"].sum()
    frequency_df["Relative Frequency"] = (frequency_df["Count"] / total_words)

    return frequency_df.sort_values(by="Count",ascending=False).reset_index(drop=True)


In [94]:
#applying the word frequency function to the different hastiness categories
very_hasty_words = df[df["Hastiness"] == "Very Hasty"]
very_formal_words = df[df["Hastiness"] == "Very Formal"]
neutral_words = df[df["Hastiness"] == "Neutral"]


very_hasty_word_freq = get_word_frequencies(very_hasty_words)
very_formal_word_freq = get_word_frequencies(very_formal_words)
neutral_word_freq = get_word_frequencies(neutral_words)


#displaying a table of the common words in the different hastiness categories

print("Very Hasty Word Frequencies:")
print(f"Total Words in Very Hasty: {very_hasty_word_freq['Count'].sum()}")
print(f"Total Unique Words in Very Hasty: {len(very_hasty_word_freq)}")
display(very_hasty_word_freq.head(10))
print()

print("Very Formal Word Frequencies:")
print(f"Total Words in Very Formal: {very_formal_word_freq['Count'].sum()}")
print(f"Total Unique Words in Very Formal: {len(very_formal_word_freq)}")
display(very_formal_word_freq.head(10))
print()

print("Neutral Word Frequencies:")
print(f"Total Words in Neutral: {neutral_word_freq['Count'].sum()}")
print(f"Total Unique Words in Neutral: {len(neutral_word_freq)}")
display(neutral_word_freq.head(10))

Very Hasty Word Frequencies:
Total Words in Very Hasty: 18748
Total Unique Words in Very Hasty: 1442


,Word,Count,Relative Frequency
0,to,803,0.042831
1,need,454,0.024216
2,and,443,0.023629
3,data,433,0.023096
4,the,339,0.018082
5,we,284,0.015148
6,n,266,0.014188
7,can,265,0.014135
8,our,255,0.013601
9,hey,254,0.013548



Very Formal Word Frequencies:
Total Words in Very Formal: 32757
Total Unique Words in Very Formal: 1558


,Word,Count,Relative Frequency
0,the,2027,0.061880
1,to,1942,0.059285
2,of,1461,0.044601
3,and,1111,0.033916
4,this,679,0.020728
5,data,652,0.019904
6,is,612,0.018683
7,our,513,0.015661
8,in,427,0.013035
9,access,424,0.012944



Neutral Word Frequencies:
Total Words in Neutral: 25236
Total Unique Words in Neutral: 1433


,Word,Count,Relative Frequency
0,to,1535,0.060826
1,and,1111,0.044024
2,our,856,0.033920
3,the,652,0.025836
4,data,547,0.021675
5,this,524,0.020764
6,access,376,0.014899
7,will,366,0.014503
8,help,355,0.014067
9,us,352,0.013948


In [95]:
#using the NGRAM API

import requests
from urllib.parse import quote

def query_ngram(word):
    url = ("https://api.ngrams.dev/eng/search"f"?query={quote(str(word))}&flags=cr")

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        return response.json()

    except requests.RequestException as error:
        print(f"API request failed for {word!r}: {error}")
        return None

wording_dict_results = query_ngram("hi") #different relTotalMatchCount for each value
display(wording_dict_results)


{'query': 'hi',
 'queryTokens': [{'text': 'hi', 'kind': 'TERM'}],
 'ngrams': [{'id': '8df4fa001ec522675b4745ec8db574f6',
   'abstract': True,
   'absTotalMatchCount': 29652674,
   'relTotalMatchCount': 1.484477740013315e-05,
   'tokens': [{'text': 'hi', 'kind': 'TERM'}]}]}

In [96]:
def get_ngram_relative_frequency(word):
    result = query_ngram(word)
    if result is None:
        return np.nan

    ngrams = result.get("ngrams", [])
    if len(ngrams) == 0:
        return 0.0

    return float(ngrams[0].get("relTotalMatchCount",0.0))

In [103]:

#querying the NGRAM baseline for the first 100 Very Hasty words

import time

very_hasty_top = very_hasty_word_freq.head(100).copy()

very_hasty_ngram_frequencies = []

for index, word in enumerate(very_hasty_top["Word"], start=1):
    frequency = get_ngram_relative_frequency(word)

    very_hasty_ngram_frequencies.append(frequency)

    print(f"{index}/100: {word} = {frequency}")

    # Small delay to avoid sending requests too rapidly
    time.sleep(0.1)

very_hasty_top["NGRAMS Relative Frequency"] = (very_hasty_ngram_frequencies)

1/100: to = 0.020195045962684514
2/100: need = 0.0002740083511912232
3/100: and = 0.023998308600294487
4/100: data = 0.0003273879521141146
5/100: the = 0.05501475193395114
6/100: we = 0.0020811429402730845
7/100: n = 0.00020321823216748255
8/100: can = 0.001538514560844412
9/100: our = 0.000914518701539268
10/100: hey = 8.549207951461519e-06
11/100: thx = 5.939578231161876e-08
12/100: for = 0.007660713853566457
13/100: access = 7.807733681251938e-05
14/100: on = 0.00513269421300439
15/100: make = 0.0005057664770330657
16/100: gotta = 1.9141928384088097e-06
17/100: so = 0.0013510108685087074
18/100: better = 0.00022134894240155874
19/100: get = 0.00031094763170706524
20/100: asap = 3.0807319303720594e-07
21/100: stuff = 1.6027104604320883e-05
22/100: how = 0.0005562113909447249
23/100: with = 0.005242616106091603
24/100: boost = 4.334172973212802e-06
25/100: us = 0.0004647327122888839
26/100: it = 0.0054602160284054424
27/100: keep = 0.00014012255629383506
28/100: wanna = 1.181167763914

In [104]:

#querying the NGRAM baseline for the first 100 Very Formal words

very_formal_top = very_formal_word_freq.head(100).copy()

very_formal_ngram_frequencies = []

for index, word in enumerate(very_formal_top["Word"], start=1):
    frequency = get_ngram_relative_frequency(word)

    very_formal_ngram_frequencies.append(frequency)

    print(f"{index}/100: {word} = {frequency}")

    # Small delay to avoid sending requests too rapidly
    time.sleep(0.1)

very_formal_top["NGRAMS Relative Frequency"] = (very_formal_ngram_frequencies)

1/100: the = 0.05501475193395114
2/100: to = 0.020195045962684514
3/100: of = 0.033448675537160015
4/100: and = 0.023998308600294487
5/100: this = 0.004047237281990258
6/100: data = 0.0003273879521141146
7/100: is = 0.008423684383745337
8/100: our = 0.000914518701539268
9/100: in = 0.017455072858922904
10/100: access = 7.807733681251938e-05
11/100: request = 7.976774215882444e-05
12/100: customer = 2.8927292406745155e-05
13/100: analysis = 0.0001945704998275812
14/100: a = 0.015161468752774571
15/100: for = 0.007660713853566457
16/100: by = 0.005305288744461861
17/100: will = 0.0017224095799333013
18/100: with = 0.005242616106091603
19/100: i = 0.003954687206930095
20/100: order = 0.0004547343426675492
21/100: that = 0.008148557087083645
22/100: marketing = 4.329986823115776e-05
23/100: team = 5.257587902776954e-05
24/100: thereby = 3.8817636336982576e-05
25/100: purpose = 0.00018457364909302992
26/100: enhance = 1.4060188772636827e-05
27/100: formally = 1.0019823771995218e-05
28/100: 

In [ ]:
#querying the NGRAM baseline for the first 100 Neutral words

neutral_top = neutral_word_freq.head(100).copy()

neutral_ngram_frequencies = []

for index, word in enumerate(neutral_top["Word"], start=1):
    frequency = get_ngram_relative_frequency(word)

    neutral_ngram_frequencies.append(frequency)

    print(f"{index}/100: {word} = {frequency}")

    # Small delay to avoid sending requests too rapidly
    time.sleep(0.1)

neutral_top["NGRAMS Relative Frequency"] = (neutral_ngram_frequencies)

1/100: to = 0.020195045962684514
2/100: and = 0.023998308600294487
3/100: our = 0.000914518701539268
4/100: the = 0.05501475193395114
5/100: data = 0.0003273879521141146
6/100: this = 0.004047237281990258
7/100: access = 7.807733681251938e-05
8/100: will = 0.0017224095799333013
9/100: help = 0.00018203414598503623
10/100: us = 0.0004647327122888839
11/100: by = 0.005305288744461861
12/100: we = 0.0020811429402730845
13/100: customer = 2.8927292406745155e-05
14/100: i = 0.003954687206930095
15/100: for = 0.007660713853566457
16/100: improve = 4.8163583509585986e-05
17/100: with = 0.005242616106091603
18/100: like = 0.000616485273044775
19/100: marketing = 4.329986823115776e-05
20/100: is = 0.008423684383745337
21/100: can = 0.001538514560844412
22/100: team = 5.257587902776954e-05
23/100: in = 0.017455072858922904
24/100: on = 0.00513269421300439
25/100: information = 0.00037573878572852616
26/100: of = 0.033448675537160015
27/100: im = 1.2297543689071e-05
28/100: better = 0.00022134894

In [ ]:
#calculating the frequency differences and z-scores

def calculate_frequency_differences(frequency_df):
    result_df = frequency_df.copy()

    #calculating the difference from the baseline relative frequency
    result_df["Frequency Difference"] = (result_df["Relative Frequency"]- result_df["NGRAMS Relative Frequency"])

    #calculating the mean and standard deviation of the differences
    difference_mean = result_df["Frequency Difference"].mean()
    difference_std = result_df["Frequency Difference"].std(ddof=0)

    #calculating the z-score
    if difference_std == 0:
        result_df["Z Score"] = 0
    else:
        result_df["Z Score"] = (result_df["Frequency Difference"] - difference_mean) / difference_std

    return result_df

In [106]:
very_hasty_results = calculate_frequency_differences(very_hasty_top)

very_formal_results = calculate_frequency_differences(very_formal_top)

neutral_results = calculate_frequency_differences(neutral_top)

In [108]:
print("Very Hasty Table:")
display(very_hasty_results.sort_values("Z Score", ascending=False).head(20).reset_index(drop=True))

print("Very Formal Table:")
display(very_formal_results.sort_values("Z Score", ascending=False).head(20).reset_index(drop=True))

print("Neutral Table:")
display(neutral_results.sort_values("Z Score", ascending=False).head(20).reset_index(drop=True))

Very Hasty Table:


,Word,Count,Relative Frequency,NGRAMS Relative Frequency,Frequency Difference,Z Score
0,need,454,0.024216,2.740084e-04,0.023942,2.873121
1,data,433,0.023096,3.273880e-04,0.022768,2.703329
2,to,803,0.042831,2.019505e-02,0.022636,2.684198
3,n,266,0.014188,2.032182e-04,0.013985,1.432466
4,hey,254,0.013548,8.549208e-06,0.013540,1.368022
5,we,284,0.015148,2.081143e-03,0.013067,1.299668
6,our,255,0.013601,9.145187e-04,0.012687,1.244657
7,can,265,0.014135,1.538515e-03,0.012596,1.231547
8,thx,235,0.012535,5.939578e-08,0.012535,1.222618
9,access,214,0.011415,7.807734e-05,0.011336,1.049261


Very Formal Table:


,Word,Count,Relative Frequency,NGRAMS Relative Frequency,Frequency Difference,Z Score
0,to,1942,0.059285,0.020195,0.039090,6.643142
1,data,652,0.019904,0.000327,0.019577,2.977768
2,this,679,0.020728,0.004047,0.016681,2.433858
3,our,513,0.015661,0.000915,0.014746,2.070405
4,access,424,0.012944,0.000078,0.012866,1.717165
5,request,401,0.012242,0.000080,0.012162,1.584957
6,customer,399,0.012181,0.000029,0.012152,1.583038
7,of,1461,0.044601,0.033449,0.011152,1.395349
8,analysis,344,0.010502,0.000195,0.010307,1.236534
9,is,612,0.018683,0.008424,0.010259,1.227582


Neutral Table:


,Word,Count,Relative Frequency,NGRAMS Relative Frequency,Frequency Difference,Z Score
0,to,1535,0.060826,0.020195,0.040631,4.609907
1,our,856,0.033920,0.000915,0.033005,3.660371
2,data,547,0.021675,0.000327,0.021348,2.208788
3,and,1111,0.044024,0.023998,0.020026,2.044183
4,this,524,0.020764,0.004047,0.016717,1.632098
5,access,376,0.014899,0.000078,0.014821,1.396070
6,help,355,0.014067,0.000182,0.013885,1.279505
7,us,352,0.013948,0.000465,0.013484,1.229500
8,will,366,0.014503,0.001722,0.012781,1.141972
9,customer,318,0.012601,0.000029,0.012572,1.116002
